In [0]:
CREATE OR REPLACE VIEW databricks_ws.gold.mart_movie_performance AS
SELECT
    f.movie_id,
    m.title,
    t.release_year AS year_release_date,
    t.decade,
    f.budget,
    f.revenue,
    f.popularity,
    f.vote_average,
    f.vote_count,

    CASE
        WHEN f.budget > 0 AND f.revenue > 0
        THEN ROUND(f.revenue / f.budget, 2)
    END AS revenue_multiple,

    CASE
        WHEN f.budget > 0 AND f.revenue > 0
        THEN CASE WHEN f.revenue >= f.budget * 2 THEN 1 ELSE 0 END
    END AS reached_2x,

    CASE
        WHEN f.budget IS NULL OR f.budget <= 0 THEN 'Sin datos'
        WHEN f.budget < 10000000 THEN 'Bajo'
        WHEN f.budget < 50000000 THEN 'Medio'
        WHEN f.budget < 100000000 THEN 'Alto'
        ELSE 'Muy alto'
    END AS budget_range

FROM databricks_ws.gold.fact_movies f
JOIN databricks_ws.gold.dim_movies m
    ON f.movie_id = m.movie_id
JOIN databricks_ws.gold.dim_time t
    ON f.time_id = t.time_id;


In [0]:

SELECT
    budget_range,
    COUNT(*) AS total_movies,
    ROUND(MEDIAN(revenue_multiple), 2) AS median_multiple,
    ROUND(AVG(reached_2x) * 100, 2) AS success_rate_2x
FROM databricks_ws.gold.mart_movie_performance
GROUP BY budget_range;

budget_range,total_movies,median_multiple,success_rate_2x
Muy alto,320,2.68,67.81
Medio,1736,1.73,45.79
Alto,667,1.86,46.18
Bajo,2002,1.3,21.68


In [0]:
SELECT
    decade,
    COUNT(*) AS total_movies,
    ROUND(MEDIAN(revenue_multiple), 2) AS median_multiple,
    ROUND(AVG(reached_2x) * 100, 2) AS success_rate_2x

FROM databricks_ws.gold.mart_movie_performance

GROUP BY decade
HAVING COUNT(reached_2x) >= 20
ORDER BY decade;


decade,total_movies,median_multiple,success_rate_2x
1940,25,3.38,68.0
1950,26,5.21,92.31
1960,71,2.94,57.75
1970,108,3.0,56.48
1980,277,2.26,51.26
1990,771,1.3,36.19
2000,2009,1.3,31.71
2010,1417,1.3,38.04
